In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Pachetul de utilitare pentru addon-uri Qiskit este o colecție de funcționalități menite să completeze fluxurile de lucru care implică unul sau mai multe addon-uri Qiskit. De exemplu, acest pachet conține funcții pentru crearea de hamiltonieni, generarea de circuite Trotter de evoluție în timp și tăierea și combinarea circuitelor cuantice.

## Instalare

Există două moduri de a instala utilitarele pentru addon-uri Qiskit: PyPI și compilarea din sursă. Se recomandă instalarea acestor pachete într-un [mediu virtual](https://docs.python.org/3.10/tutorial/venv.html) pentru a asigura separarea între dependențele pachetelor.

### Instalare din PyPI

Cel mai simplu mod de a instala pachetul de utilitare pentru addon-uri Qiskit este prin PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Instalare din sursă
<details>
<summary>
Apasă aici pentru a citi cum să instalezi acest pachet manual.
</summary>

Dacă dorești să contribui la acest pachet sau vrei să îl instalezi manual, clonează mai întâi depozitul:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

și instalează pachetul prin `pip`. Dacă plănuiești să rulezi tutorialele din depozitul pachetului, instalează și dependențele pentru notebook. Dacă plănuiești să dezvolți în depozit, instalează dependențele `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Începe să folosești utilitarele
Există mai multe module în pachetul `qiskit-addon-utils`, inclusiv unul pentru generarea de probleme în simularea sistemelor cuantice, colorarea grafurilor pentru plasarea mai eficientă a Gate-urilor într-un Circuit cuantic și tăierea circuitelor, care poate ajuta cu [backpropagarea operatorilor](/guides/qiskit-addons-obp). Secțiunile de mai jos rezumă fiecare modul. [Documentația API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) a pachetului conține, de asemenea, informații utile.

### Generarea de probleme
Conținutul modulului [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) include:

- O funcție [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), care generează o reprezentare `SparsePauliOp` conștientă de conectivitate a unui model XYZ de tip Ising:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- O funcție [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), care construiește un Circuit ce modelează evoluția în timp a unui operator dat.
- Trei obiecte diferite [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) pentru enumerarea diferitelor ordonări ale șirurilor Pauli. Acestea sunt în special utile atunci când sunt folosite împreună cu colorarea grafurilor și pot fi utilizate ca argumente atât în funcțiile `generate_xyz_hamiltonian()`, cât și `generate_time_evolution_circuit()`.

### Colorarea grafurilor
Modulul [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) este folosit pentru a colora muchiile dintr-o hartă de cuplare și pentru a utiliza această colorare în plasarea mai eficientă a Gate-urilor într-un Circuit cuantic. Scopul acestei hărți de cuplare cu muchii colorate este de a găsi un set de culori pentru muchii astfel încât nicio două muchii de aceeași culoare să nu împartă un nod comun. Pentru un QPU, acest lucru înseamnă că Gate-urile de-a lungul muchiilor de aceeași culoare (conexiuni între Qubiți) pot fi rulate simultan, iar circuitul se va executa mai rapid.

Ca exemplu rapid, poți folosi funcția [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) pentru a genera o colorare a muchiilor pentru un Circuit naiv care execută un `CZGate` de-a lungul fiecărei conexiuni de Qubiți. Fragmentul de cod de mai jos folosește harta de cuplare a Backend-ului `FakeSherbrooke`, creează acest Circuit naiv, apoi folosește funcția `auto_color_edges()` pentru a crea un Circuit echivalent mai eficient.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Tăierea circuitelor
În cele din urmă, modulul [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) conține funcții și pasaje ale Transpiler-ului pentru lucrul cu „felii" de Circuit, partiții de tip temporal ale unui [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) care se întind pe toți Qubiții. Aceste felii sunt folosite în principal pentru [backpropagarea operatorilor](/guides/qiskit-addons-obp-get-started). Principalele patru moduri în care un Circuit poate fi tăiat sunt: după tipul Gate-ului, după adâncime, după colorare sau după instrucțiuni [`Barrier`](../api/qiskit/circuit#barrier). Ieșirea acestor funcții de tăiere returnează o listă de obiecte [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). Circuitele tăiate pot fi, de asemenea, recombinate folosind funcția [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Citește [referința API](../api/qiskit-addon-utils/slicing) a modulului pentru mai multe informații.

Mai jos sunt câteva exemple despre cum să creezi aceste felii folosind următorul Circuit:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

În cazul în care nu există o modalitate clară de a exploata structura unui Circuit pentru backpropagarea operatorilor, poți împărți circuitul în felii de o anumită adâncime.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Dacă ai o strategie personalizată de tăiere, poți plasa Barrier-uri în Circuit pentru a delimita unde trebuie tăiat și poți folosi funcția [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).